In [30]:
import pandas

In [31]:
import requests

In [32]:
import pandas as pd

In [33]:
from bs4 import BeautifulSoup

In [34]:
response = requests.get("https://www.metmuseum.org/art/collection/search?showOnly=withImage%7CopenAccess&department=11&searchField=ArtistCulture&q=Gustave+Courbet")

In [35]:
response

<Response [200]>

In [36]:
html_string = response.text

In [37]:
print(html_string[:500] + "...")

<!DOCTYPE html><html lang="en" class="
				__variable_362d9d
				__variable_bfed6e
				__variable_56d10a
				__variable_1411e1
				__variable_f75c8b
				__variable_75d57f" data-sentry-component="RootLayout" data-sentry-source-file="layout.tsx"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1"/><link rel="preload" as="image" href="https://images.metmuseum.org/CRDImages/ep/mobile-large/DT1975.jpg"/><link rel="preload" as="image" href="https://images.m...


In [38]:
import requests

In [39]:
from bs4 import BeautifulSoup

In [40]:
document = BeautifulSoup(html_string, "html.parser")

In [41]:
document

<!DOCTYPE html>
<html class="__variable_362d9d __variable_bfed6e __variable_56d10a __variable_1411e1 __variable_f75c8b __variable_75d57f" data-sentry-component="RootLayout" data-sentry-source-file="layout.tsx" lang="en"><head><meta charset="utf-8"/><meta content="width=device-width, initial-scale=1" name="viewport"/><link as="image" href="https://images.metmuseum.org/CRDImages/ep/mobile-large/DT1975.jpg" rel="preload"/><link as="image" href="https://images.metmuseum.org/CRDImages/ep/mobile-large/DP-23236-001.jpg" rel="preload"/><link as="image" href="https://images.metmuseum.org/CRDImages/ep/mobile-large/DT1963.jpg" rel="preload"/><link as="image" href="https://images.metmuseum.org/CRDImages/ep/mobile-large/DT1967.jpg" rel="preload"/><link as="image" href="https://images.metmuseum.org/CRDImages/ep/mobile-large/DT1964.jpg" rel="preload"/><link data-precedence="next" href="/_next/static/css/05c2505592d96dca.css" rel="stylesheet"/><link data-precedence="next" href="/_next/static/css/f0da8

In [42]:
import requests

In [43]:
from bs4 import BeautifulSoup

In [44]:
#find all the images of the collections
document.find_all("img")

[<img class="collection-object_image__XVQPm collection-object_gridView__8kZLF" loading="eager" src="https://images.metmuseum.org/CRDImages/ep/mobile-large/DT1975.jpg"/>,
 <img class="collection-object_image__XVQPm collection-object_gridView__8kZLF" loading="eager" src="https://images.metmuseum.org/CRDImages/ep/mobile-large/DP-23236-001.jpg"/>,
 <img class="collection-object_image__XVQPm collection-object_gridView__8kZLF" loading="eager" src="https://images.metmuseum.org/CRDImages/ep/mobile-large/DT1963.jpg"/>,
 <img class="collection-object_image__XVQPm collection-object_gridView__8kZLF" loading="eager" src="https://images.metmuseum.org/CRDImages/ep/mobile-large/DT1967.jpg"/>,
 <img class="collection-object_image__XVQPm collection-object_gridView__8kZLF" loading="eager" src="https://images.metmuseum.org/CRDImages/ep/mobile-large/DT1964.jpg"/>,
 <img class="collection-object_image__XVQPm collection-object_gridView__8kZLF" loading="lazy" src="https://images.metmuseum.org/CRDImages/ep/mob

In [45]:
#to find the title of the painting
document.find("h1").text

'Search The Collection'

In [46]:
import requests
from bs4 import BeautifulSoup
import csv

# Function to extract artwork details from the Met's collection URL
def scrape_artwork_info(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Extracting the title
    title_tag = soup.find('h1', {'id': 'artwork__title'})
    title = ''
    if title_tag:
        title = title_tag.find('span', {'class': 'artwork__title__inset'})
        if title:
            title = title.get_text(strip=True)

# Extracting the artist name using the correct class
    artist = ''
    artist_tag = soup.find('span', {'class': 'artwork__artist__name'})
    if artist_tag:
        artist = artist_tag.get_text(strip=True)
    
    # Extracting the image URL
    image_url = ''
    image_tag = soup.find('div', {'id': 'artwork__image__wrapper'})
    if image_tag:
        image = image_tag.find('img')
        if image and 'src' in image.attrs:
            image_url = image.attrs['src']

    # Exclude entries with unwanted artist qualifiers
    exclude_keywords = ["Attributed to", "Style of", "Copy after"]
    if any(keyword in artist for keyword in exclude_keywords):
        print(f"Excluded due to artist qualifier: {artist}")
        return None  # Skip this entry if it matches exclude criteria

    # Exclude entries not matching the desired artist name
    if artist != "Gustave Courbet":
        print(f"Excluded: {artist} (Not Gustave Courbet)")
        return None  # Skip this entry if it's not by Gustave Courbet
    
    # Debugging outputs (can be commented out later)
    print(f"Title: {title}")
    print(f"Artist: {artist}")
    print(f"Image URL: {image_url}")
    
    return {
        'title': title,
        'artist': artist,
        'image_url': image_url
    }

In [47]:
# Function to extract artwork URLs from a search page
def get_artwork_links(search_url):
    response = requests.get(search_url)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Extract all 'href' attributes from 'figure' tags
    artwork_links = []
    for figure in soup.find_all('figure'):
        a_tag = figure.find('a')  # Find the anchor tag inside the figure
        if a_tag and 'href' in a_tag.attrs:
            artwork_links.append(a_tag['href'])
    
    # Convert relative links to absolute URLs
    base_url = "https://www.metmuseum.org"
    artwork_links = [base_url + link for link in artwork_links if link.startswith('/')]
    
    return artwork_links

In [48]:
# URL of the search page for "Gustave Courbet"
search_url = "https://www.metmuseum.org/art/collection/search?q=Gustave+Courbet"

# Get dynamically extracted URLs
urls = get_artwork_links(search_url)
print(f"Found {len(urls)} artwork links.")

# Scrape data from the dynamically extracted URLs
artworks_data = []
for url in urls:
    artwork_info = scrape_artwork_info(url)
    if artwork_info:  # Exclude None entries
        artworks_data.append(artwork_info)

Found 40 artwork links.
Title: After the Hunt
Artist: Gustave Courbet
Image URL: https://collectionapi.metmuseum.org/api/collection/v1/iiif/436007/796196/main-image
Title: Young Ladies of the Village
Artist: Gustave Courbet
Image URL: https://collectionapi.metmuseum.org/api/collection/v1/iiif/438820/796200/main-image
Title: The Calm Sea
Artist: Gustave Courbet
Image URL: https://collectionapi.metmuseum.org/api/collection/v1/iiif/436005/796206/main-image
Title: The Young Bather
Artist: Gustave Courbet
Image URL: https://collectionapi.metmuseum.org/api/collection/v1/iiif/436003/1819005/main-image
Title: Marine: The Waterspout
Artist: Gustave Courbet
Image URL: https://collectionapi.metmuseum.org/api/collection/v1/iiif/436006/796198/main-image
Title: Louis Gueymard (1822–1880) as Robert le Diable
Artist: Gustave Courbet
Image URL: https://collectionapi.metmuseum.org/api/collection/v1/iiif/436015/796208/main-image
Title: Jo, La Belle Irlandaise
Artist: Gustave Courbet
Image URL: https://co

In [49]:
# Remove None entries from the data
artworks_data = [artwork for artwork in artworks_data if artwork is not None]


# Save data to CSV file
csv_filename = r"C:\Users\re99n\TheMetMuseum_GustaveCourbet\theMetMuseum_GustaveCourbet_Metadata_Fixed.csv"
fieldnames = ["title", "artist", "image_url"]

with open(csv_filename, "w", newline="", encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for artwork in artworks_data:
        writer.writerow(artwork)

print(f"Data has been saved to {csv_filename}")

Data has been saved to C:\Users\re99n\TheMetMuseum_GustaveCourbet\theMetMuseum_GustaveCourbet_Metadata_Fixed.csv


In [63]:
# The CSV file contains special characters and might have a BOM (Byte Order Mark),
# so we use 'utf-8-sig' encoding to ensure proper reading of the file without encoding issues.
df = pd.read_csv('C:/Users/re99n/TheMetMuseum_GustaveCourbet/theMetMuseum_GustaveCourbet_Metadata_Fixed.csv', encoding='utf-8-sig')

# Save it again as UTF-8-sig
df.to_csv('C:/Users/re99n/TheMetMuseum_GustaveCourbet/theMetMuseum_GustaveCourbet_Metadata_Final_fixed.csv', encoding='utf-8-sig', index=False)